In [18]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table


In [19]:
df = pd.read_csv("../../data/dosm_job_demand.csv")

In [20]:
# -------------------------
# 1. Create date column
# -------------------------
quarter_map = {
    1: "01-01",
    2: "04-01",
    3: "07-01",
    4: "10-01"
}

df["Quarter"] = df["Quarter"].astype(int)

df["date"] = pd.to_datetime(
    df["Year"].astype(str) + "-" + df["Quarter"].map(quarter_map)
)

In [21]:
# -------------------------
# 2. Rename columns
# -------------------------
df = df.rename(columns={
    "Skills": "skill_level",
    "Economic Activity": "sector",
    "Sub-economic Activity": "subsector",
    "Jobs ('000)": "job_available",
    "Filled Jobs ('000)": "job_filled",
    "Vacancies ('000)": "job_vacancy",
    "Jobs Created ('000)": "job_created"
})

In [22]:
# -------------------------
# 3. Convert numeric columns
# remove commas and multiply by 1000
# -------------------------
num_cols = [
    "job_available",
    "job_filled",
    "job_vacancy",
    "job_created"
]

for col in num_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float) * 1000
    )

In [23]:
# -------------------------
# 4. Select final columns
# -------------------------
df_final = df[[
    "date",
    "skill_level",
    "sector",
    "subsector",
    "job_available",
    "job_filled",
    "job_vacancy",
    "job_created"
]]
df_final["sector"] = df_final["sector"].replace({
    "Mining & Quarrying": "Mining and Quarrying"
})
df_final.head()

C:\Users\Asus\AppData\Local\Temp\ipykernel_19660\76038711.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final["sector"] = df_final["sector"].replace({


,date,skill_level,sector,subsector,job_available,job_filled,job_vacancy,job_created
0,2025-01-01,Skilled,Agriculture,Agriculture,28800.0,24400.0,4400.0,100.0
1,2025-01-01,Skilled,Mining and Quarrying,Mining & Quarrying,23500.0,23400.0,100.0,100.0
2,2025-01-01,Skilled,Manufacturing,"Food processing, beverages & tobacco products",46100.0,43600.0,2400.0,400.0
3,2025-01-01,Skilled,Manufacturing,"Textiles, wearing apparel & leather products",10400.0,9800.0,600.0,0.0
4,2025-01-01,Skilled,Manufacturing,"Wood products, furniture, paper products & pri...",36400.0,34400.0,2000.0,300.0


In [26]:
# -------------------------
# 5. Aggregate by skill_level and sector
# -------------------------
df_final = df_final.groupby(['date','skill_level', 'sector'])[[
    'job_available',
    'job_filled',
    'job_vacancy',
    'job_created'
]].sum().reset_index()

df_final.head(20)

,date,skill_level,sector,job_available,job_filled,job_vacancy,job_created
0,2018-01-01,Low-skilled,Agriculture,35200.0,26700.0,8500.0,100.0
1,2018-01-01,Low-skilled,Construction,50200.0,45200.0,5000.0,300.0
2,2018-01-01,Low-skilled,Manufacturing,158300.0,140100.0,18200.0,0.0
3,2018-01-01,Low-skilled,Mining and Quarrying,10000.0,9900.0,100.0,0.0
4,2018-01-01,Low-skilled,Services,889100.0,878800.0,10300.0,1700.0
5,2018-01-01,Semi-skilled,Agriculture,408000.0,392100.0,15900.0,1900.0
6,2018-01-01,Semi-skilled,Construction,1109800.0,1099700.0,10100.0,2400.0
7,2018-01-01,Semi-skilled,Manufacturing,1639700.0,1578100.0,61500.0,4300.0
8,2018-01-01,Semi-skilled,Mining and Quarrying,47400.0,47200.0,200.0,200.0
9,2018-01-01,Semi-skilled,Services,2065200.0,2043300.0,22000.0,5400.0


In [27]:
write_table(df_final, "sc_bronze", "dosm_jobdemand")

Table sc_bronze.dosm_jobdemand written successfully.
